# 03 Verify: scores, realism diagnostics, verif bridge

Scores the matched dataset with block-bootstrap confidence intervals, checks whether the forecasts are smoother than the observations, exports to MET Norway's `verif` format, and optionally builds a lagged ensemble.

## What this notebook does and does not show

Does: the `ScoreTable` (bias, MAE, RMSE per model, variable, lead and stratum) saved as parquet and jsonl, a MAE pivot by lead and model, one hand-run block bootstrap, variance ratio and Q-Q and tail-conditional diagnostics, a `verif` text export and plot, and a fair-CRPS lagged ensemble.

Does not: use the network anywhere. Nothing here says which model is better in general; the sample is one short window over a few dozen stations. See the closing cell of `04_report.ipynb`.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from afv.config import load_config
from afv.dataset import read_dataset, read_manifest

CONFIG = Path(os.environ.get("AFV_CONFIG", "configs/default.yaml"))
cfg = load_config(CONFIG)
out_dir = cfg.get_path("results_dir")
matched = read_dataset(out_dir / "matched.parquet")
man = read_manifest(out_dir / "matched.parquet")
primary = cfg["interp"].get("primary", {})
print(CONFIG, "->", cfg["name"])
print(man.get("rows"), "rows | dataset hash", man.get("dataset_hash"), "| stations", man.get("stations"))
print(matched.groupby(["model", "variable"]).size().unstack(fill_value=0))

## Score table

`score_table` uses the primary interpolation per variable, scores every stratum in `cfg.strata`, and attaches a 90 percent block-bootstrap interval that resamples whole days (`valid_date`) rather than rows, because station errors are correlated in time. The dataset hash and config hash travel with the table.

In [ ]:
from afv.scores import score_table, threshold_scores, block_bootstrap, mae

st = score_table(matched, cfg, dataset_hash=man.get("dataset_hash", ""))
st.to_parquet(out_dir / "scores.parquet")
st.to_jsonl(out_dir / "scores.jsonl")
for model, rows in st.rows.groupby("model"):
    sub = type(st)(rows=rows.reset_index(drop=True), dataset_hash=st.dataset_hash,
                   config_hash=st.config_hash, afv_version=st.afv_version)
    sub.to_parquet(out_dir / f"scores_{model}.parquet")
print(len(st.rows), "score rows | dataset", st.dataset_hash, "| config", st.config_hash, "| afv", st.afv_version)
st.rows.head()

In [ ]:
allr = st.rows[(st.rows["stratum"] == "all") & (st.rows["metric"] == "mae")]
allr.pivot_table(index=["variable", "lead_h"], columns="model", values="value", aggfunc="first").round(3)

One cell of that table done by hand with the generic `block_bootstrap`, to show what the interval means: the same day blocks are resampled `n_resamples` times and the score recomputed each time.

In [ ]:
from afv.scores import add_strata

model0 = cfg["models"][0]
m = add_strata(matched)
sub = m[(m["model"] == model0) & (m["variable"] == "t2m") & (m["lead_h"] == 24) & (m["interp"] == primary.get("t2m"))]
bs = cfg["bootstrap"]
point, lo, hi = block_bootstrap(sub, mae, block_col=bs["block_col"], n=int(bs["n_resamples"]), ci=float(bs["ci"]), seed=int(bs["seed"]))
print(f"MAE t2m lead 24 h, {model0}: {point:.3f} K [{lo:.3f}, {hi:.3f}] from {sub['valid_date'].nunique()} day blocks, n={len(sub)}")

Threshold scores (ETS, frequency bias, POD, FAR at 1, 5 and 10 mm per 24 h) only apply to `tp24`, which is disabled in the default config.

In [ ]:
if "tp24" in set(matched["variable"]):
    thr = threshold_scores(matched[matched["variable"] == "tp24"], list(cfg["thresholds_mm_24h"]))
    thr.to_csv(out_dir / "threshold_scores_tp24.csv", index=False)
    display(thr.round(3))
else:
    print("tp24 is not in the matched table (disabled in the config); threshold scores skipped")

## Realism diagnostics

A model can win MAE by forecasting the conditional mean (Bonavita 2024; Rasp et al. 2024). Three checks sit next to the scores: the variance ratio `std(fcst) / std(obs)` per lead (below 1 means smoother than the observations), the spatial activity of the raw field per lead, and the error restricted to the observed upper tail.

In [ ]:
from afv.diagnostics import variance_ratio, qq, tail_conditional, activity_vs_lead

vr = variance_ratio(matched, primary)
vr.to_csv(out_dir / "diag_variance_ratio.csv", index=False)
tail = tail_conditional(matched, float(cfg["diagnostics"]["tail_quantile"]))
tail.to_csv(out_dir / "diag_tail.csv", index=False)
fields = sorted(Path(cfg.get_path("fields_dir")).glob("*.nc"))
act = activity_vs_lead(fields, cfg["diagnostics"].get("activity_variable", "t2m"))
act.to_csv(out_dir / "diag_activity.csv", index=False)
print(len(vr), "variance-ratio rows,", len(tail), "tail rows,", len(act), "activity rows")

In [ ]:
import matplotlib.pyplot as plt

for var, r in vr.groupby("variable"):
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    for model, g in r.groupby("model"):
        g = g.sort_values("lead_h")
        ax.plot(g["lead_h"], g["variance_ratio"], marker="o", label=model)
    ax.axhline(1.0, color="k", lw=0.8, ls="--")
    ax.set_xlabel("lead time (h)")
    ax.set_ylabel("std(fcst) / std(obs)")
    ax.set_title(f"{var}: below 1 means the forecast is smoother than the observations")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.show()

In [ ]:
var = "t2m"
fig, ax = plt.subplots(figsize=(4.6, 4.6))
for model in sorted(matched["model"].unique()):
    q = qq(matched, model, var, interp=primary.get(var))
    ax.plot(q["obs_q"], q["fcst_q"], marker=".", ms=3, label=model)
lims = ax.get_xlim()
ax.plot(lims, lims, "k--", lw=0.8)
ax.set_xlabel("observed quantile (K)")
ax.set_ylabel("forecast quantile (K)")
ax.set_title(f"Q-Q, {var}: tails below the line are underforecast")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

In [ ]:
# Observed values above the 90th percentile per variable. Small n is shown, not hidden.
tail.sort_values(["variable", "model", "lead_h"]).round(3)

## verif bridge

MET Norway's `verif` (Nipen et al. 2023) takes one text file per forecast system with the columns `date hour leadtime location lat lon altitude obs fcst`. It aligns the systems itself and silently drops any sample that is missing in one of them. That is the same same-sample rule `pairwise_complete` applies explicitly, so the numbers should agree; if they do not, the alignment is the first place to look.

In [ ]:
from afv.verif_bridge import to_verif_text, verif_available, run_verif

verif_dir = out_dir / "verif"
var = "t2m"
files = []
for model in cfg["models"]:
    p = to_verif_text(matched, model, var, verif_dir / f"{model}_{var}.txt", interp=primary.get(var))
    n_rows = sum(1 for _ in p.open(encoding="utf-8")) - 1
    files.append(p)
    print(p, n_rows, "rows")
print("verif on PATH:", verif_available())

In [ ]:
from IPython.display import Image, display

if verif_available():
    png = verif_dir / "mae.png"
    proc = run_verif(files, "mae", out_png=png)
    print("return code", proc.returncode)
    if proc.stdout:
        print(proc.stdout[-2000:])
    if proc.stderr:
        print(proc.stderr[-2000:])
    if png.exists():
        display(Image(filename=str(png)))
else:
    print("verif is not installed (pip install verif). The command would be:")
    print("verif", " ".join(str(f) for f in files), "-m mae -f mae.png")

## Lagged ensemble (optional)

Brenowitz et al. 2025 point out that a deterministic model already provides an ensemble for free: the forecasts from successive initialisations that are valid at the same time. Member 0 is the newest run, member 1 the run before it, and so on. The spread between them is a crude but honest uncertainty estimate and it makes a probabilistic score (CRPS) and a spread-skill check possible without an ensemble product.

With initialisations at 00 and 12 UTC the members are 12 h apart, so `step_h` is 12 here rather than the config default of 6, which would need 6 hourly initialisations. Three members at lead L, L+12 and L+24 need L+24 within the 72 h range.

In [ ]:
from afv.baselines import lagged_ensemble, members_matrix
from afv.scores import crps_ensemble, spread_skill

le = cfg["baselines"]["lagged_ensemble"]
n_members = int(le["n_members"])
step_h = 12  # inits are 00 and 12 UTC in the default config; step_h 6 needs 6 hourly inits
for model in cfg["models"]:
    lag = lagged_ensemble(matched, model, n_members=n_members, step_h=step_h, interp=primary.get("t2m"))
    lag = lag[lag["variable"] == "t2m"] if not lag.empty else lag
    if lag.empty:
        print(f"{model}: no complete lagged sets (needs {n_members} successive inits valid at the same time)")
        continue
    X, y = members_matrix(lag)
    spread, skill = spread_skill(X, y)
    print(f"{model}: {X.shape[0]} samples x {X.shape[1]} members | fair CRPS {crps_ensemble(X, y):.3f} K "
          f"| spread {spread:.3f} K | RMSE of the mean {skill:.3f} K | MAE of member 0 {mae(X[:, 0], y):.3f} K")

A lagged ensemble mixes lead times, so its CRPS is not comparable with a proper ensemble at a single lead. It is reported as a baseline for the deterministic scores, not as a forecast product.